# Aula 02 — Laboratório
## O que o dado mostra antes do modelo

# Bloco 0 — O painel da Aula 01 na mesa


**Meta do bloco:** todo mundo com um painel mensal carregado e inspecionado.

Se você salvou o painel da Aula 01 em CSV, suba o arquivo (`painel_aula01.csv`).
Se não salvou, siga assim mesmo: o notebook recoleta as séries pelas mesmas APIs.

Ao painel da Aula 01 entram **duas séries novas**, e elas são o material do Bloco 2:

| Código SGS | Série |
|---|---|
| 24363 | IBC-Br — índice de atividade econômica, **sem** ajuste sazonal |
| 24364 | IBC-Br — **com** ajuste sazonal, feito pelo Banco Central |

In [ ]:
# Bloco 0 — rode esta célula e confira a saída.
import os, sys, platform
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 14)
plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# As mesmas cores dos slides. Use-as nos seus graficos.
AZUL, LARANJA, VERDE, BORDO = "#2a78d6", "#eb6834", "#1baf7a", "#6F1D2B"

print("Python :", sys.version.split()[0], "|", platform.system())
print("pandas :", pd.__version__)
print("numpy  :", np.__version__)
print("\nAmbiente pronto.")

In [ ]:
# Bloco 0 — de onde vem o painel de hoje. Leia, nao precisa alterar.
#
# ORDEM DE PREFERENCIA:
#   1. o painel que VOCE montou na Aula 01, salvo em CSV;
#   2. se nao houver arquivo, o notebook recoleta pelas mesmas APIs;
#   3. se nao houver rede, PLANO B com dados sinteticos (inventados).
#
# Para usar o seu painel, deixe na pasta Notebooks o CSV que voce exportou
# na Aula 01 (index = mes, colunas ipca/selic/dolar).

CAMINHO_PAINEL = "painel_aula01.csv"
MODO_OFFLINE = False


def voltar_ao_modo_real():
    global MODO_OFFLINE
    MODO_OFFLINE = False
    print("Modo real reativado. A proxima coleta vai tentar a API.")


def sgs(codigo, inicio="01/01/2017"):
    url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
    r = requests.get(url, params={"formato": "json", "dataInicial": inicio}, timeout=120)
    r.raise_for_status()
    df = pd.DataFrame(r.json())
    df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
    df["valor"] = pd.to_numeric(df["valor"], errors="coerce")
    return df.set_index("data")["valor"].sort_index()


def painel_sintetico(semente=7):
    rng = np.random.default_rng(semente)
    idx = pd.period_range("2017-01", "2025-12", freq="M")
    n, mes = len(idx), idx.month

    # --- IPCA: base + efeito do mes do ano + componente persistente ---------
    efeito_mes = {1: 0.30, 2: 0.32, 3: 0.20, 4: 0.12, 5: 0.06, 6: -0.04,
                  7: -0.06, 8: -0.10, 9: 0.00, 10: 0.10, 11: 0.16, 12: 0.24}
    regime = np.zeros(n)                                  # AR(1): inflacao tem inercia
    for i in range(1, n):
        regime[i] = 0.70 * regime[i - 1] + rng.normal(0, 0.11)
    ipca = (0.30 + np.array([efeito_mes[m] for m in mes])
            + regime + rng.normal(0, 0.08, n))
    covid = (idx >= pd.Period("2020-03", "M")) & (idx <= pd.Period("2020-07", "M"))
    ipca[covid] -= 0.55                                   # deflacao dos meses fechados
    aperto = (idx >= pd.Period("2021-04", "M")) & (idx <= pd.Period("2022-06", "M"))
    ipca[aperto] += 0.55                                  # choque de custos

    # --- Selic: regra de reacao a inflacao acumulada (causalidade reversa) ---
    ipca12 = pd.Series(ipca).rolling(12).sum().bfill().to_numpy()
    selic = np.zeros(n)
    selic[0] = 10.0
    for i in range(1, n):
        alvo = 2.0 + 1.3 * ipca12[i - 1]
        selic[i] = 0.80 * selic[i - 1] + 0.20 * alvo + rng.normal(0, 0.10)
    selic[covid] -= 2.0                                   # corte de emergencia

    # --- Dolar: passeio aleatorio com salto em marco/2020 -------------------
    dolar = 3.2 + np.cumsum(rng.normal(0.003, 0.06, n))
    dolar[idx >= pd.Period("2020-03", "M")] += 1.50

    # --- IBC-Br: tendencia x fator sazonal, com tombo da pandemia ----------
    fator = {1: 0.93, 2: 0.90, 3: 1.02, 4: 0.98, 5: 1.03, 6: 0.99,
             7: 1.02, 8: 1.04, 9: 1.00, 10: 1.05, 11: 1.03, 12: 1.11}
    tend = 100 * (1 + 0.0022 * np.arange(n))
    tombo = np.ones(n)
    for k, p in enumerate(["2020-03", "2020-04", "2020-05", "2020-06", "2020-07"]):
        tombo[idx == pd.Period(p, "M")] = [0.93, 0.78, 0.82, 0.88, 0.93][k]
    ibc_dessaz = tend * tombo + rng.normal(0, 0.6, n)
    ibc = ibc_dessaz * np.array([fator[m] for m in mes]) + rng.normal(0, 0.5, n)

    painel = pd.DataFrame(
        {"ipca": ipca.round(2), "selic": selic.round(2), "dolar": dolar.round(4),
         "ibc": ibc.round(2), "ibc_dessaz": ibc_dessaz.round(2)}, index=idx)
    painel.iloc[-1, painel.columns.get_loc("ipca")] = np.nan   # defasagem de divulgacao
    painel.iloc[-2:, painel.columns.get_loc("ibc")] = np.nan
    painel.iloc[-2:, painel.columns.get_loc("ibc_dessaz")] = np.nan
    return painel


def painel_por_api():
    series = {"ipca": 433, "selic": 432, "dolar": 1, "ibc": 24363, "ibc_dessaz": 24364}
    colunas = {}
    for nome, codigo in series.items():
        s = sgs(codigo)
        if nome == "selic":
            s = s.resample("ME").mean()      # taxa: media do mes
        elif nome == "dolar":
            s = s.resample("ME").last()      # cotacao: fechamento do mes
        s.index = s.index.to_period("M")
        colunas[nome] = s
    return pd.DataFrame(colunas).round(4)


def carregar_painel(caminho=CAMINHO_PAINEL):
    global MODO_OFFLINE
    if os.path.exists(caminho):
        df = pd.read_csv(caminho, index_col=0)
        df.index = pd.PeriodIndex(df.index, freq="M")
        print(f"[SEU PAINEL] {caminho}: {df.shape[0]} meses, colunas {list(df.columns)}")
        faltam = [c for c in ("ibc", "ibc_dessaz") if c not in df.columns]
        if faltam:
            print("Faltam as series do dia", faltam, "- buscando so elas.")
            try:
                extra = painel_por_api()[faltam]
                df = df.join(extra, how="outer")
            except Exception as e:
                print("[FALHOU]", type(e).__name__, "-", e, "-> completando pelo Plano B.")
                df = df.join(painel_sintetico()[faltam], how="outer")
        return df
    if not MODO_OFFLINE:
        try:
            df = painel_por_api()
            print(f"[REAL] painel recoletado da API: {df.shape[0]} meses.")
            return df
        except Exception as e:
            print(f"[FALHOU] coleta: {type(e).__name__} - {e}")
            MODO_OFFLINE = True
    df = painel_sintetico()
    print("=" * 72)
    print("[PLANO B] PAINEL SINTETICO (dados inventados).")
    print("A mecanica do dia e identica; as conclusoes economicas, nao valem.")
    print("=" * 72)
    return df


print("Carga pronta. Rode a proxima celula.")

In [ ]:
# Bloco 0 — carregue o painel e olhe para ele antes de qualquer conta.
painel = carregar_painel()

display(painel.tail(6))
print("\nFormato:", painel.shape)
print("\nAusentes por coluna:")
print(painel.isna().sum())

> ### Checkpoint
> Você só avança quando `painel` existe, tem índice mensal e pelo menos as colunas
> `ipca`, `selic`, `dolar`, `ibc` e `ibc_dessaz`.

# Bloco 1 — O primeiro olhar


Resumo numérico **descreve**; gráfico **mostra**. As duas coisas não são
intercambiáveis, e quem começa pelo `describe()` costuma nunca ver a série.

A pergunta que organiza o bloco: *esta série está em **nível** ou em **variação**,
e o que cada uma responde?*

In [ ]:
# Bloco 1 — EXEMPLO RESOLVIDO. Rode e compare as duas saidas.
display(painel[["ipca", "selic", "dolar"]].describe().round(2))

fig, eixos = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
x = painel.index.to_timestamp()          # PeriodIndex nao vai direto ao matplotlib
for ax, col, cor, rot in zip(eixos, ["ipca", "selic", "dolar"],
                             [LARANJA, AZUL, VERDE],
                             ["IPCA (% no mes)", "Selic (% a.a.)", "Dolar (R$)"]):
    ax.plot(x, painel[col], color=cor, linewidth=1.6)
    ax.set_ylabel(rot, fontsize=9)
eixos[-1].set_xlabel("")
fig.suptitle("As tres series do painel", fontsize=12)
plt.tight_layout()
plt.show()

print("O describe() acima descreve alguma dessas tres formas? Qual ele esconde?")

In [ ]:
# Bloco 1 — TAREFA 1.1
# Desenhe o dolar de duas maneiras, lado a lado:
#   (a) em NIVEL, como esta no painel;
#   (b) em VARIACAO percentual mes a mes.
# Dica: serie.pct_change() * 100

dolar_var = ...   # <-- COMPLETE

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.6))
a1.plot(x, painel["dolar"], color=VERDE, linewidth=1.6)
a1.set_title("Dolar em nivel (R$)", fontsize=11)
a2.plot(x, dolar_var, color=VERDE, linewidth=1.2)
a2.axhline(0, color="#5C5F6B", linewidth=0.8)
a2.set_title("Dolar em variacao (% no mes)", fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# Bloco 1 — TAREFA 1.2
# A media descreve a serie? Compare a media do dolar no periodo INTEIRO
# com a media dos ULTIMOS 24 meses, e com a dos 24 PRIMEIROS.

print("media do periodo inteiro :", ...)   # <-- COMPLETE
print("media dos 24 primeiros   :", ...)   # <-- COMPLETE
print("media dos 24 ultimos     :", ...)   # <-- COMPLETE

In [ ]:
# Bloco 1 — TAREFA 1.3
# Em quantos meses o IPCA passou de 0,50%? E em que anos eles se concentram?
# Dica: painel.index.year devolve o ano de cada linha.

acima = painel["ipca"] > 0.50
print("meses acima de 0,50%:", ...)          # <-- COMPLETE: quantos
print()
print(...)                                   # <-- COMPLETE: contagem por ano

> **Pare e responda:** Para prever "quanto o dólar vai custar no mês que
> vem", você usaria a série em nível ou em variação? E para responder "o dólar
> está caro?"
>
> *sua resposta:*

# Bloco 2 — Sazonalidade e tendência


Toda série econômica mensal carrega um calendário dentro: décimo terceiro, férias,
safra, volta às aulas, Natal. Isso é **sazonalidade** — um padrão que se repete no
mesmo mês, todo ano, e que **não é notícia**.

O Banco Central publica o IBC-Br em duas versões: a **bruta** (24363) e a **com
ajuste sazonal** (24364). Você vai fazer o ajuste na mão e comparar com o oficial.

In [ ]:
# Bloco 2 — EXPERIMENTO 2.0 (rode e observe)
# A media do IPCA em cada mes do ano, ao longo de toda a amostra.
por_mes = painel["ipca"].groupby(painel.index.month).mean()

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.bar(por_mes.index, por_mes.values, color=LARANJA)
ax.axhline(painel["ipca"].mean(), color=BORDO, linestyle="--", linewidth=1.2,
           label="media geral")
ax.set_xticks(range(1, 13))
ax.set_xlabel("mes do ano"); ax.set_ylabel("% no mes")
ax.set_title("IPCA medio por mes do ano", fontsize=12)
ax.legend(frameon=False)
plt.show()

print("Diferenca entre o mes mais alto e o mais baixo:",
      round(por_mes.max() - por_mes.min(), 3), "p.p.")

In [ ]:
# Bloco 2 — TAREFA 2.1
# A media esconde a dispersao. Faca um BOXPLOT do IPCA por mes do ano.
# Dica: monte uma lista com os valores de cada mes e passe ao ax.boxplot().

grupos = [... for m in range(1, 13)]   # <-- COMPLETE: os valores do mes m, sem NaN

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.boxplot(grupos, tick_labels=list(range(1, 13)))
ax.set_xlabel("mes do ano"); ax.set_ylabel("% no mes")
ax.set_title("IPCA por mes do ano — distribuicao", fontsize=12)
plt.show()

In [ ]:
# Bloco 2 — TAREFA 2.2
# Agora o IBC-Br. Desenhe no MESMO grafico a serie bruta e a oficial dessazonalizada.

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(x, ..., color=AZUL, linewidth=1.4, label="IBC-Br bruto (24363)")      # <-- COMPLETE
ax.plot(x, ..., color=BORDO, linewidth=1.8, label="IBC-Br dessaz. (24364)")  # <-- COMPLETE
ax.set_title("Atividade economica: com e sem o calendario dentro", fontsize=12)
ax.legend(frameon=False)
plt.show()

In [ ]:
# Bloco 2 — TAREFA 2.3
# Faca voce mesmo o ajuste sazonal, pelo metodo classico da razao para a media movel:
#   1. tendencia  = media movel de 12 meses, centrada
#   2. razao      = serie / tendencia
#   3. fator[m]   = media das razoes daquele mes do ano
#   4. dessaz     = serie / fator[mes]

tendencia = painel["ibc"].rolling(12, center=True).mean()
razao = painel["ibc"] / tendencia
fator = ...                                    # <-- COMPLETE: media da razao por mes do ano
meu_dessaz = ...                               # <-- COMPLETE: serie / fator do mes

comp = pd.DataFrame({"oficial": painel["ibc_dessaz"], "meu": meu_dessaz}).dropna()
print("correlacao com o oficial :", round(comp["oficial"].corr(comp["meu"]), 4))
print("erro medio absoluto      :", round((comp["oficial"] - comp["meu"]).abs().mean(), 3))
display(fator.round(3))

In [ ]:
# Bloco 2 — TAREFA 2.4
# Duas leituras da mesma atividade economica:
#   (a) variacao contra o mes anterior, na serie BRUTA
#   (b) variacao contra o MESMO MES do ano anterior (12 meses atras), na serie bruta
# Dica: .pct_change() e .pct_change(12)

var_mes = ...    # <-- COMPLETE
var_12m = ...    # <-- COMPLETE

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(x, var_mes, color="#B4B2AC", linewidth=1.0, label="contra o mes anterior")
ax.plot(x, var_12m, color=BORDO, linewidth=1.8, label="contra o mesmo mes do ano anterior")
ax.axhline(0, color="#5C5F6B", linewidth=0.8)
ax.set_ylabel("%"); ax.set_title("IBC-Br bruto: duas variacoes", fontsize=12)
ax.legend(frameon=False)
plt.show()

print("desvio-padrao da variacao mensal :", round(var_mes.std(), 2))
print("desvio-padrao da variacao em 12m :", round(var_12m.std(), 2))

> ### Responda
>
> **1.** Em qual mês do ano o IPCA é sistematicamente mais alto na sua amostra?
> Isso é informação econômica ou é calendário?
>
> *sua resposta:*
>
> **2.** Um jornal publica: *"atividade econômica cresce 8% em dezembro"*, usando o
> IBC-Br **bruto** contra novembro. Onde está o erro, e qual número ele deveria ter usado?
>
> *sua resposta:*
>
> **3.** Se você jogar a série **bruta** dentro de um modelo de previsão, ele vai
> aprender a sazonalidade. Isso é um problema? Quando sim, quando não?
>
> *sua resposta:*

# Bloco 3 — Quebras estruturais

Modelo de machine learning aprende do passado e supõe que o futuro se parece com ele.
**Quebra estrutural** é o momento em que essa suposição deixa de valer: a relação entre
as variáveis mudou, e nenhum volume de dados de antes conserta isso.

Aqui você não vai "olhar e achar" a quebra. Vai **medir**.

In [ ]:
# Bloco 3 — EXPERIMENTO 3.0 (rode e observe)
# Media e desvio em janelas moveis de 24 meses. Se a serie fosse estavel, as duas linhas andariam de lado.
m24 = painel["dolar"].rolling(24).mean()
s24 = painel["dolar"].rolling(24).std()

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
a1.plot(x, painel["dolar"], color="#B4B2AC", linewidth=1.0, label="dolar")
a1.plot(x, m24, color=BORDO, linewidth=2, label="media movel de 24 meses")
a1.legend(frameon=False); a1.set_ylabel("R$")
a2.plot(x, s24, color=AZUL, linewidth=1.8)
a2.set_ylabel("desvio-padrao movel")
fig.suptitle("A media desta serie fica parada em algum lugar?", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Bloco 3 — TAREFA 3.1
# A correlacao entre a VARIACAO do dolar e o IPCA e estavel no tempo?
# Calcule a correlacao em janela movel de 24 meses e desenhe.
# Dica: serie_a.rolling(24).corr(serie_b)

corr_movel = ...   # <-- COMPLETE

fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(x, corr_movel, color=LARANJA, linewidth=1.8)
ax.axhline(0, color="#5C5F6B", linewidth=0.9)
ax.set_ylim(-1, 1); ax.set_ylabel("correlacao")
ax.set_title("Correlacao movel (24 meses): variacao do dolar x IPCA", fontsize=12)
plt.show()

print("correlacao na amostra inteira:", round(dolar_var.corr(painel["ipca"]), 3))
print("minimo da movel:", round(corr_movel.min(), 3),
      "| maximo:", round(corr_movel.max(), 3))

In [ ]:
# Bloco 3 — TAREFA 3.2
# Corte a amostra em marco de 2020 e compare os dois pedacos.

corte = pd.Period("2020-03", freq="M")
antes = painel[painel.index < corte]
depois = painel[painel.index >= corte]

for col in ["ipca", "selic", "dolar"]:
    ...   # <-- COMPLETE: imprima col, media antes, media depois e a diferenca

In [ ]:
# Bloco 3 — TAREFA 3.3
# Ache a quebra PELOS DADOS, sem olhar o calendario: varra todas as datas de corte possiveis e fique com a que mais separa as duas medias do dolar.
# (E a intuicao por tras do teste de Chow, sem a estatistica formal.)

serie = painel["dolar"].dropna()
melhor_data, melhor_gap = None, 0.0

for i in range(12, len(serie) - 12):          # 12 meses minimos de cada lado
    esq, dir_ = serie.iloc[:i], serie.iloc[i:]
    gap = ...                                  # <-- COMPLETE: distancia entre as medias
    if gap > melhor_gap:
        melhor_gap, melhor_data = gap, serie.index[i]

print("quebra sugerida pelos dados:", melhor_data, "| salto de", round(melhor_gap, 3))

> **Pare e responda:** Você vai treinar um modelo para prever o dólar do mês que vem, e tem dados desde 2017. Usa a amostra inteira, só o período depois da quebra ou a amostra inteira com uma variável marcando o regime? Defenda a sua escolha em duas linhas — e diga o que você perde nela.

> *sua resposta:*

# Bloco 4 — Correlação espúria e endogeneidade

Duas séries que sobem com o tempo se correlacionam. Isso não é economia: é aritmética.
Este bloco tem duas partes, e elas se parecem, mas são problemas diferentes:

- **espúria** — a correlação não existe de verdade; é efeito de tendência comum;
- **endógena** — a correlação existe, mas a seta aponta para o outro lado (ou para os dois).

In [ ]:
# Bloco 4 — EXPERIMENTO 4.1 (rode e olhe o absurdo)
# Duas series INVENTADAS, sem nenhuma relacao entre si: dois passeios aleatorios.
rng = np.random.default_rng(2026)
a = np.cumsum(rng.normal(0, 1, 120))
b = np.cumsum(rng.normal(0, 1, 120))

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(a, color=AZUL, linewidth=1.6, label="serie A (inventada)")
ax.plot(b, color=LARANJA, linewidth=1.6, label="serie B (inventada)")
ax.legend(frameon=False); ax.set_title("Nenhuma relacao entre as duas", fontsize=12)
plt.show()

print("correlacao entre A e B:", round(np.corrcoef(a, b)[0, 1], 3))

# E agora a mesma coisa, 2000 vezes:
meio, forte = 0, 0
for _ in range(2000):
    u = np.cumsum(rng.normal(0, 1, 120))
    v = np.cumsum(rng.normal(0, 1, 120))
    c = abs(np.corrcoef(u, v)[0, 1])
    meio += c > 0.5
    forte += c > 0.7
print(f"|correlacao| > 0,5 em {meio/20:.1f}% dos pares SEM relacao alguma.")
print(f"|correlacao| > 0,7 em {forte/20:.1f}% dos pares SEM relacao alguma.")

> **Responda antes de seguir:** por que duas séries sem nenhuma relação exibem
> correlação alta com tanta frequência? O que elas têm em comum?
>
> *sua resposta:*

In [ ]:
# Bloco 4 — TAREFA 4.2
# Compare a matriz de correlacao do painel em NIVEL e em PRIMEIRA DIFERENCA.
# Dica: painel.diff() calcula a variacao de cada coluna.

cols = ["ipca", "selic", "dolar", "ibc_dessaz"]
print("EM NIVEL")
display(...)          # <-- COMPLETE
print("EM PRIMEIRA DIFERENCA")
display(...)          # <-- COMPLETE

In [ ]:
# Bloco 4 — funcao pronta. Uma regressao minima, so com numpy.

def regressao(y, x, nome="modelo"):
    """OLS de y contra x (com intercepto). Devolve dict com coeficiente, R2 e t."""
    d = pd.DataFrame({"y": y, "x": x}).dropna()
    X = np.column_stack([np.ones(len(d)), d["x"].to_numpy(float)])
    yv = d["y"].to_numpy(float)
    beta, *_ = np.linalg.lstsq(X, yv, rcond=None)
    resid = yv - X @ beta
    gl = len(d) - 2
    s2 = resid @ resid / gl
    var_beta = s2 * np.linalg.inv(X.T @ X)[1, 1]
    r2 = 1 - (resid @ resid) / ((yv - yv.mean()) ** 2).sum()
    t = beta[1] / np.sqrt(var_beta)
    print(f"{nome:28s} | coef = {beta[1]:+8.4f} | R2 = {r2:5.3f} | t = {t:+7.2f} | n = {len(d)}")
    return {"coef": beta[1], "r2": r2, "t": t, "n": len(d)}

print("Funcao regressao() pronta.")

In [ ]:
# Bloco 4 — TAREFA 4.3
# Regrida a inflacao contra a Selic. ANTES de rodar, escreva aqui o sinal que voce espera:
#
#   sinal esperado: ____________   (juro alto derruba inflacao => negativo?)
#
# Use a inflacao ACUMULADA EM 12 MESES, que e a leitura que o Copom persegue.
# Dica: soma movel de 12 meses -> serie.rolling(12).sum()

ipca12 = ...        # <-- COMPLETE

regressao(ipca12, painel["selic"], "ipca 12m ~ selic")
regressao(painel["ipca"], painel["selic"], "ipca do mes ~ selic")

> **O sinal bateu com o que você escreveu?** Se a Selic sobe e a inflação medida
> *sobe junto*, a leitura ingênua seria "juro alto causa inflação". Por que essa
> conclusão está errada?
>
> *sua resposta:*

In [ ]:
# Bloco 4 — TAREFA 4.4
# Quem vem antes? Calcule a correlacao entre o IPCA de hoje e a Selic deslocada de -6 a +6 meses. Se o pico estiver no FUTURO da Selic, e a Selic que responde.
# Dica: serie.shift(k) empurra a serie k periodos para a frente.

for k in range(-12, 7):
    c = ...    # <-- COMPLETE: correlacao entre painel["ipca"] e a Selic deslocada de k
    marca = "#" * int(abs(c) * 40)
    print(f"k = {k:+d}  corr = {c:+.3f}  {marca}")

> ### Responda
>
> **1.** No experimento 4.1, em que percentual das simulações apareceu |correlação| > 0,7
> entre séries sem relação alguma? Escreva o número que **você** obteve.
>
> *sua resposta:*
>
> **2.** Qual das correlações do seu painel sobreviveu à primeira diferença, e qual
> desapareceu? O que isso sugere sobre cada uma?
>
> *sua resposta:*
>
> **3.** Em uma frase: qual a diferença entre uma correlação **espúria** e uma relação
> **endógena**?
>
> *sua resposta:*

# Bloco 5 — Viés de seleção e o diagnóstico do seu painel

Falta a última armadilha, e é a mais fácil de cair sem perceber: **escolher a amostra com base no que você quer explicar**. Depois disso, o desafio em dupla.

In [ ]:
# Bloco 5 — EXPERIMENTO 5.1 (rode e observe)
# Um banco quer saber se RENDA prevê CALOTE. Ele tem o historico dos clientes, ou seja, so de quem ELE APROVOU. Vamos simular o mundo inteiro e ver o que o
# banco enxerga de dentro da propria carteira.
rng = np.random.default_rng(5)
n = 5000

renda = rng.normal(0, 1, n)                 # o que o banco observa
outro = rng.normal(0, 1, n)                 # o que o banco tambem usa: garantia, historico
score = renda + outro                       # a nota de credito
latente = -(0.9 * renda + 0.9 * outro) + rng.normal(0, 1, n)
calote = (latente > 0.8).astype(int)        # quem de fato nao paga
aprovado = score > 0.5                      # o banco so empresta acima da nota de corte

print("No mundo inteiro (todos os candidatos):")
print("   corr(renda, calote) =", round(np.corrcoef(renda, calote)[0, 1], 3),
      "| calote =", f"{calote.mean():.1%}", "| n =", n)
print("Na carteira do banco (so os aprovados):")
print("   corr(renda, calote) =", round(np.corrcoef(renda[aprovado], calote[aprovado])[0, 1], 3),
      "| calote =", f"{calote[aprovado].mean():.1%}", "| n =", int(aprovado.sum()))
print()
print("corr(renda, outro) no mundo inteiro :", round(np.corrcoef(renda, outro)[0, 1], 3))
print("corr(renda, outro) entre aprovados  :",
      round(np.corrcoef(renda[aprovado], outro[aprovado])[0, 1], 3))

> **Responda antes de seguir:** no mundo inteiro, renda e garantia são independentes (correlação ≈ 0). Entre os aprovados, elas ficam **fortemente correlacionadas**, e negativamente. Ninguém mexeu nos dados. De onde veio essa correlação?
>
> *sua resposta:*
>
> E o que acontece com a conclusão do banco sobre o poder preditivo da renda?
>
> *sua resposta:*

In [ ]:
# Bloco 5 — TAREFA 5.2
# A mesma armadilha no SEU painel: selecionar um pedaco da amostra e reportar o numero como se fosse geral.
# Calcule a correlacao entre a variacao do dolar e o IPCA em tres recortes: (a) amostra inteira   (b) so ate 2019   (c) so de 2021 em diante

base = pd.DataFrame({"d_dolar": dolar_var, "ipca": painel["ipca"]}).dropna()

print("amostra inteira :", ...)   # <-- COMPLETE
print("so ate 2019     :", ...)   # <-- COMPLETE
print("2021 em diante  :", ...)   # <-- COMPLETE

## Desafio em dupla — o diagnóstico do seu painel


Agora sem passo a passo. Vocês escrevem, no espaço abaixo, o **laudo do painel de vocês**: o que ele tem dentro, e o que isso proíbe de fazer com ele.

### O que entregar

1. **Três achados**, cada um com o número que o sustenta e a célula que o produziu.
   Um deles tem de ser sobre sazonalidade, outro sobre quebra ou estabilidade.
2. **Uma armadilha**: uma correlação do seu painel que você **não** reportaria, e por quê.
3. **A ficha de diagnóstico** preenchida (logo abaixo).
4. **Uma decisão para a Semana 04**: que transformação vocês vão levar para a modelagem
   — nível, primeira diferença, variação em 12 meses, série dessazonalizada — e por quê.

### Como isso é avaliado

| Critério | O que estamos olhando |
|---|---|
| Diagnóstico | Os achados vêm de um número medido, não de impressão sobre o gráfico |
| Tratamento | A transformação escolhida responde a um problema que vocês mostraram |
| Ceticismo | A armadilha identificada é real, e a explicação está correta |
| Leitura | Alguém que não esteve na aula entende o laudo sem vocês do lado |

In [ ]:
# Bloco 5 — ESPACO DE TRABALHO (achado 1)





In [ ]:
# Bloco 5 — ESPACO DE TRABALHO (achado 2)





In [ ]:
# Bloco 5 — ESPACO DE TRABALHO (achado 3 + a armadilha)





---
## 📋 Ficha de diagnóstico do painel — preencha antes de entregar

Uma linha por série do seu painel.

| | Série 1 | Série 2 | Série 3 |
|---|---|---|---|
| **Nome da série** | | | |
| **Nível ou variação?** | | | |
| **Tem sazonalidade?** (qual mês, e quanto) | | | |
| **A média descreve a série?** (sim/não, e por quê) | | | |
| **Tem quebra?** (data que você mediu) | | | |
| **Correlação que some ao diferenciar** | | | |
| **Transformação escolhida para a Semana 04** | | | |
| **O que esta série NÃO permite afirmar** | | | |

### Uma última pergunta, para responder em duas linhas

Escolha **um par** de variáveis do seu painel que se correlacionam. Escreva a frase
causal que alguém escreveria a partir dessa correlação — e depois escreva por que
o seu próprio painel **não** sustenta essa frase.

*sua resposta:*

---

## Entrega

`Arquivo → Fazer download → Download .ipynb` e envie pelo canal da disciplina,
nomeado como `aula02_SOBRENOME1_SOBRENOME2.ipynb`.

## Semana 04

Começa a modelagem preditiva: treino e teste em série temporal (que não se corta
ao acaso), a primeira regressão com `statsmodels`, e a pergunta que vai organizar
o bloco inteiro — *este modelo prevê, ou só decorou?*